# 3.3 — Retreino com Inputs Reduzidos: SVR, DT, RF, XGBoost

Fase 3 da Etapa 3: retreina as 4 arquiteturas sklearn para k ∈ {4, 5, 6} inputs e os 3 outputs.  
Hiperparâmetros idênticos à Etapa 2.1 (D-E3-06). Total: 36 runs no experimento `reduzido`.

**Pré-requisitos:** `selecao/3.2_subconjuntos.csv` gerado pela Etapa 3.2.

## Seção 1 — Imports e configuração

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

RANDOM_STATE  = 42
BASE_DIR      = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
PROCESSED_DIR = BASE_DIR / "ARTEFATOS" / "ETAPA_0" / "processed"
SELECAO_DIR   = BASE_DIR / "ARTEFATOS" / "ETAPA_3" / "3.2_FEAT_SELECTION"
ARTIFACTS_DIR = BASE_DIR / "ARTEFATOS" / "ETAPA_3" / "3.3_REDUZIDO"
MLFLOW_URI    = f"file:///{BASE_DIR}/mlruns"
EXPERIMENT    = "reduzido"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print("MLflow URI:", mlflow.get_tracking_uri())
print("Experimento:", EXPERIMENT)

## Seção 2 — Carga dos dados e subconjuntos de features

In [ ]:
X_train = np.load(PROCESSED_DIR / "X_train.npy")
X_val   = np.load(PROCESSED_DIR / "X_val.npy")
X_test  = np.load(PROCESSED_DIR / "X_test.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_val   = np.load(PROCESSED_DIR / "y_val.npy")
y_test  = np.load(PROCESSED_DIR / "y_test.npy")

scaler_y_min   = np.load(PROCESSED_DIR / "scaler_y_min.npy")
scaler_y_scale = np.load(PROCESSED_DIR / "scaler_y_scale.npy")

# Ordem das colunas no X processado (mesma do preprocessing)
FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
FEAT_IDX      = {f: i for i, f in enumerate(FEATURE_NAMES)}

# Carregar subconjuntos S_4, S_5, S_6
subsets_df = pd.read_csv(SELECAO_DIR / "3.2_subconjuntos.csv")
subsets = {}
for _, row in subsets_df.iterrows():
    k = int(row["k"])
    feats = [f.strip() for f in row["features"].split(",")]
    subsets[k] = feats

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)
print()
for k, feats in sorted(subsets.items()):
    idx = [FEAT_IDX[f] for f in feats]
    print(f"  S_{k}: {feats}  →  colunas {idx}")

## Seção 3 — Grids de hiperparâmetros (idênticos à Etapa 2.1 — D-E3-06)

In [ ]:
OUTPUT_NAMES = ["M_CH3OH", "x_CH3OH", "ET"]

MODELS = [
    (
        "SVR",
        SVR(),
        {
            "C":       [0.1, 1, 10, 100],
            "epsilon": [0.001, 0.01, 0.1],
            "kernel":  ["rbf"],
        },
    ),
    (
        "DT",
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {
            "max_depth":         [3, 5, 10, None],
            "min_samples_split": [2, 5, 10],
        },
    ),
    (
        "RF",
        RandomForestRegressor(random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100, 200],
            "max_depth":    [5, 10, None],
        },
    ),
    (
        "XGBoost",
        XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
        {
            "n_estimators":  [100, 200, 300],
            "max_depth":     [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1],
        },
    ),
]


def denorm_y(y_norm, output_idx):
    return y_norm * scaler_y_scale[output_idx] + scaler_y_min[output_idx]


def compute_metrics(y_true_norm, y_pred_norm, output_idx):
    y_true = denorm_y(y_true_norm, output_idx)
    y_pred = denorm_y(y_pred_norm, output_idx)
    return {
        "r2":  r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
    }

## Seção 4 — Loop de treinamento (k × modelo × output)

In [ ]:
results = []

for k in [4, 5, 6]:
    feat_list = subsets[k]
    col_idx   = [FEAT_IDX[f] for f in feat_list]

    X_tr_k  = X_train[:, col_idx]
    X_val_k = X_val[:,   col_idx]
    X_te_k  = X_test[:,  col_idx]

    for model_name, estimator, param_grid in MODELS:
        for output_idx, output_name in enumerate(OUTPUT_NAMES):
            print(f"\n{'='*60}")
            print(f"  k={k}  |  {model_name}  |  {output_name}")
            print(f"{'='*60}")

            y_tr = y_train[:, output_idx]
            y_v  = y_val[:,   output_idx]
            y_te = y_test[:,  output_idx]

            gs = GridSearchCV(
                estimator,
                param_grid,
                cv=5,
                scoring="r2",
                n_jobs=-1,
                refit=True,
            )
            gs.fit(X_tr_k, y_tr)
            best = gs.best_estimator_
            print(f"  Melhores params: {gs.best_params_}")

            metrics_train = compute_metrics(y_tr, best.predict(X_tr_k),  output_idx)
            metrics_val   = compute_metrics(y_v,  best.predict(X_val_k), output_idx)
            metrics_test  = compute_metrics(y_te, best.predict(X_te_k),  output_idx)
            print(f"  R² train={metrics_train['r2']:.4f}  val={metrics_val['r2']:.4f}  test={metrics_test['r2']:.4f}")

            # Artefato .pkl
            art_path = ARTIFACTS_DIR / model_name / output_name / f"k{k}"
            art_path.mkdir(parents=True, exist_ok=True)
            pkl_path = art_path / "model.pkl"
            with open(pkl_path, "wb") as f:
                pickle.dump(best, f)

            # Run MLflow
            with mlflow.start_run(run_name=f"{model_name}_{output_name}_k{k}"):
                mlflow.set_tag("model",      model_name)
                mlflow.set_tag("output",     output_name)
                mlflow.set_tag("k",          str(k))
                mlflow.set_tag("fase",       "retreino")

                mlflow.log_param("arquitetura",      model_name)
                mlflow.log_param("output",           output_name)
                mlflow.log_param("k",                k)
                mlflow.log_param("inputs_selecionados", str(feat_list))
                mlflow.log_param("metodo_selecao",   "shap_global_topk")
                mlflow.log_param("random_state",     RANDOM_STATE)
                for hp, v in gs.best_params_.items():
                    mlflow.log_param(hp, v)

                for split, m in [("train", metrics_train), ("val", metrics_val), ("test", metrics_test)]:
                    mlflow.log_metric(f"{split}_r2",  m["r2"])
                    mlflow.log_metric(f"{split}_mae", m["mae"])
                    mlflow.log_metric(f"{split}_mse", m["mse"])

                mlflow.log_artifact(str(pkl_path))

            results.append({
                "model":             model_name,
                "output":            output_name,
                "k":                 k,
                "inputs_selecionados": feat_list,
                **{f"train_{m}": v for m, v in metrics_train.items()},
                **{f"val_{m}":   v for m, v in metrics_val.items()},
                **{f"test_{m}":  v for m, v in metrics_test.items()},
            })

print("\n=== Treinamento sklearn concluído ===")
print(f"Total de runs: {len(results)} (esperado: 36)")

## Seção 5 — Tabela de resultados

In [ ]:
results_df = pd.DataFrame(results)

print("=== R² test set por (k, modelo, output) ===")
pivot = results_df.pivot_table(
    index=["k", "model"],
    columns="output",
    values="test_r2",
).round(4)
display(pivot)

print("\n=== Tabela completa ===")
display(
    results_df[[
        "k", "model", "output",
        "train_r2", "val_r2", "test_r2",
        "train_mae", "val_mae", "test_mae",
    ]]
    .sort_values(["k", "model", "output"])
    .reset_index(drop=True)
    .round(4)
)

In [ ]:
# Validação: 36 runs, nenhum R² NaN ou negativo
assert len(results_df) == 36, f"Esperado 36 runs, obtido {len(results_df)}"
assert results_df["test_r2"].notna().all(), "NaN encontrado em test_r2"
neg = results_df[results_df["test_r2"] < 0]
if len(neg) > 0:
    print("AVISO: runs com R² negativo:", neg[["k", "model", "output", "test_r2"]].to_dict("records"))
else:
    print("Validação OK: 36 runs, nenhum R² NaN ou negativo.")